In [1]:
path = "data_base/bbc_text_cls_adapted.csv"

In [2]:
import pandas as pd
import numpy as np
import random
import math
from sklearn.metrics import confusion_matrix

In [3]:
data = pd.read_csv(filepath_or_buffer = path)

In [4]:
labels = data["labels"].unique()

In [5]:
X = data["text"]
y = data["labels"]

In [6]:
if y.dtype == object or str:
    flag = True
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    y = le.fit_transform(y)
    y = pd.DataFrame(y)

In [7]:
from sklearn.model_selection import train_test_split

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.5, random_state = 1, stratify = y)

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words='english')

X_train_transformed = vectorizer.fit_transform(X_train)
X_test_transformed = vectorizer.transform(X_test)

In [10]:
indices_treino = y_train.index
columns = data.iloc[indices_treino, 2:]

In [11]:
columns = columns.apply(le.transform)

In [12]:
matrix = np.array(columns)

In [13]:
matrix = np.array([list(dict.fromkeys(lista)) for lista in matrix], dtype=object)

In [14]:
len_ = 0
for _ in matrix:
    if 1 < len(_):
        len_+=1

In [15]:
labels_trans = le.transform(labels)

In [16]:
individual = random.choices(labels_trans, k=len_)

In [17]:
individual = np.array(individual)

In [18]:
from sklearn.naive_bayes import MultinomialNB
naive_bayes_mult = MultinomialNB()

In [19]:
from sklearn.metrics import r2_score

In [20]:
def removal_function(individual,n_remove):
    # individual = list(individual)
    indexes = []
    length = len(individual)
    num = length-n_remove
    index = random.randint(0,num)
    indexes.append(index)
    indexes.append(index+1)
    indexes.append(index+2)

    return(indexes)

In [21]:
def determinator(tiny_list):
    choice = random.choice(tiny_list)
    return(choice)

In [22]:
def insertion_function(indexes,individual,matrix): 
    valores_transformados = [determinator(matrix[v]) for v in indexes]
    for index, _ in enumerate(indexes):
        individual[_] = valores_transformados[index]
        
    return(individual)
    

In [23]:
def make_list_proba(df_):
    total_proba_ = df_[0].prod()
    total_proba_2 = df_[1].prod()
    total_proba_3 = df_[2].prod()
    total_proba_4 = df_[3].prod()
    total_proba_5 = df_[4].prod()
    list_proba_ = []
    list_proba_.append(total_proba_)
    list_proba_.append(total_proba_2)
    list_proba_.append(total_proba_3)
    list_proba_.append(total_proba_4)
    list_proba_.append(total_proba_5)
    return(list_proba) 

In [24]:
def fitness_function(individual,matrix,X_train_transformed):
    matrix_ = matrix.copy()
    df_ = pd.DataFrame()
    cont = 0
    for index,values in enumerate(matrix_):
        if len(values) > 1:
            matrix_[index] = individual[cont]
            cont+=1

    matrix_ = [item[0] if isinstance(item, list) else item for item in matrix_]

    naive_bayes_mult.fit(X_train_transformed,matrix_)
    naive_bayes_mult.predict(X_train_transformed)
    df_ = naive_bayes_mult.predict_log_proba(X_train_transformed)
    list_before_sum = [max(i) for i in df_]
    fitness = sum(list_before_sum)
    
    return(fitness)

In [25]:
def implementation_function(individual,n_remove,X_train_transformed,matrix,k):
    best_fitness = float('-inf')
    best_individual = []
    fitness = 0
    euler = math.e
    
    for _ in range(k):
        indexes = removal_function(individual,n_remove)
        individual = insertion_function(indexes,individual,matrix)
        fitness = fitness_function(individual,matrix,X_train_transformed)
        # print(fitness)
        # print(best_fitness)
        
        if fitness > best_fitness:
            best_individual = individual
            best_fitness = fitness
        else:
            e_dif = euler ** (best_fitness-fitness)# VERIFICAR
            if e_dif > round(random.uniform(1,e_dif*2 ),2): # Ex: 54.32 random.randint(0,(e_dif*2)):
                best_individual = individual
                best_fitness = fitness

    return(best_fitness,best_individual)

In [26]:
best_fitness,best_individual = implementation_function(individual,3,X_train_transformed,matrix,1000)

In [27]:
print(best_fitness)
print(best_individual)

-1207.7563438073528
[0 1 4 ... 3 3 0]


In [28]:
matrix_ = matrix.copy()

df_ = pd.DataFrame()
cont = 0
for index,values in enumerate(matrix_):
    if len(values) > 1:
        matrix_[index] = best_individual[cont]
        cont+=1

matrix_ = [item[0] if isinstance(item, list) else item for item in matrix_]

In [29]:
if flag == True:
    matrix_ = le.inverse_transform(matrix_)
    y_train = le.inverse_transform(y_train)

C:\Users\pedro\anaconda3\Lib\site-packages\sklearn\preprocessing\_label.py:151: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [30]:
confusion_matrix(y_train, matrix_)

array([[52, 53, 49, 61, 40],
       [30, 43, 41, 45, 34],
       [49, 27, 46, 43, 43],
       [52, 47, 54, 52, 50],
       [45, 33, 28, 43, 52]])